# Flood Response Routing: Risk-Aware Optimization

**Independent portfolio project · Fictional case study · Python + Gurobi (or SciPy HiGHS)**

**Business question:** A city response vehicle leaves a central hub, delivers to five essential service sites, and returns to base. How much longer should it drive to avoid potentially exposed streets? What if specific roads are closed?

**Deliverables:** Network overview, a mixed-integer routing model with MTZ subtour elimination, distance-only vs. risk-aware route comparison, disrupted-road scenarios, and a sensitivity chart.

> **Data ethics:** All 25 nodes, 48 streets, hazards, and service locations are invented for demonstration. The risk scores are relative, uncalibrated indices—not real flood probabilities. This notebook is independent of any university starter notebook or assignment material.

## 1. Setup and load a synthetic city

`nodes.csv` stores map coordinates and which stops are required. `streets.csv` stores undirected road segments with distances in meters and synthetic exposure indices in `[0,1]`. `closures.csv` describes disruption scenarios.

The implementation selects **Gurobi automatically if installed**, otherwise uses the **SciPy HiGHS** MILP solver. You can explicitly choose `backend='scipy'` when a Gurobi license is unavailable.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.routing import load_city, plot_network, solve_route

# Use 'gurobi' for a licensed Gurobi installation, 'scipy' for no license.
BACKEND = 'auto'
DATA_DIR = Path('data')
nodes, streets, closures = load_city(DATA_DIR)
print(f'{len(nodes)} nodes | {len(streets)} undirected streets | '
      f'{nodes.is_required.sum()} required stops | '
      f'{closures.scenario.nunique()} closure scenarios')
display(nodes.loc[nodes.is_required == 1, ['node_id', 'label', 'kind']])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_network(nodes, streets, title='Synthetic response network', ax=ax)
plt.show()

## 2. Mathematical programming formulation

Define `V` as all nodes, `A` as directed orientations of available streets, `R` as all required nodes (including `DEPOT`), `dᵢⱼ` as distance, and `rᵢⱼ` as relative flood-exposure index.

Binary decision `xᵢⱼ=1` when the vehicle travels directly from `i` to `j`. The risk weight `λ` is a user-chosen, **illustrative preference parameter**.

$$\min\ \sum_{(i,j)\in A} (d_{ij}+\lambda d_{ij}r_{ij})x_{ij}$$

**Constraints** (implemented in `src/routing.py`):

1. **Balanced flow:** $\sum_j x_{ij}=\sum_j x_{ji}$ for all nodes.
2. **Required service:** $\sum_j x_{ij}=1$ for each required node, including the depot.
3. **Optional junctions:** $\sum_j x_{ij}\leq1$ for all non-required nodes.
4. **No repeated undirected street:** $x_{ij}+x_{ji}\leq1$ for each street.
5. **MTZ elimination:** $u_i-u_j+(|V|-1)x_{ij}\leq |V|-2$ for arcs excluding the depot; $1\leq u_i\leq|V|-1$.
6. $x_{ij}\in\{0,1\}$.

**How MTZ works:** If a selected arc goes from `i` to `j`, then `uⱼ ≥ uᵢ+1`. A disconnected directed cycle would require node order to keep increasing around a closed loop, which is impossible. For an unselected arc, the big-M coefficient makes the constraint inactive. The depot is excluded from MTZ because the route must return there.

## 3. Solve and compare two policies

- **Shortest route:** `λ=0`; only total distance matters.
- **Risk-aware route:** `λ=0.5`; exposure contributes to the objective.

We calculate **risk exposure = Σ(length × relative risk index)**, in risk-weighted meters. This is *not* a probabilistic safety estimate.

In [ ]:
fast = solve_route(nodes, streets, closures, risk_weight=0, backend=BACKEND)
safer = solve_route(nodes, streets, closures, risk_weight=0.5, backend=BACKEND)
comparison = pd.DataFrame([
    {'Strategy':'Shortest distance', 'Travel distance (m)':fast.distance_m,
     'Risk exposure (risk × m)':round(fast.risk_exposure_m,2), 'Solver':fast.backend},
    {'Strategy':'Risk-aware (λ=0.5)', 'Travel distance (m)':safer.distance_m,
     'Risk exposure (risk × m)':round(safer.risk_exposure_m,2), 'Solver':safer.backend}
])
display(comparison)
print('Shortest route:\n', ' → '.join(fast.route))
print('\nRisk-aware route:\n', ' → '.join(safer.route))
print(f'\nExtra travel: {safer.distance_m-fast.distance_m:.0f} m '
      f'({100*(safer.distance_m/fast.distance_m-1):.1f}%)')
print(f'Exposure reduction: {100*(1-safer.risk_exposure_m/fast.risk_exposure_m):.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
plot_network(nodes, streets, fast.route, title='Shortest-distance route', ax=axes[0])
plot_network(nodes, streets, safer.route, title='Risk-aware route (λ = 0.5)', ax=axes[1])
plt.show()

## 4. Storm disruptions: re-optimize when roads close

Road closures remove those street segments **before optimization**. Each of the five runs is separately optimized; it is not just a route patched manually after the fact.

In [ ]:
experiments = [('normal', 0), ('normal', .5),
               ('flash_flood', 0), ('flash_flood', .5),
               ('localized_outage', .5)]
scenario_rows = []
for scenario, weight in experiments:
    sol = solve_route(nodes, streets, closures, scenario=scenario,
                      risk_weight=weight, backend=BACKEND)
    scenario_rows.append({'Scenario':scenario, 'Risk weight':weight,
                          'Distance (m)':sol.distance_m,
                          'Exposure (risk × m)':round(sol.risk_exposure_m, 2),
                          'Runtime (s)':round(sol.runtime_s, 3)})
scenario_table = pd.DataFrame(scenario_rows)
display(scenario_table)

In [ ]:
flash = solve_route(nodes, streets, closures, scenario='flash_flood',
                    risk_weight=.5, backend=BACKEND)
blocked = set(closures.loc[closures.scenario == 'flash_flood', 'street_id'])
open_streets = streets[~streets.street_id.isin(blocked)]
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
plot_network(nodes, streets, safer.route, title='Normal roads — risk-aware', ax=axes[0])
plot_network(nodes, open_streets, flash.route, title='Closed roads — re-optimized', ax=axes[1])
plt.show()
print('Blocked streets:', ', '.join(sorted(blocked)))

## 5. Sensitivity analysis — how much risk aversion changes the decision

Re-solve the model over several nonnegative values of `λ`. Different weights may yield the same route: this is expected for a discrete optimization problem. Plotting exposure on the horizontal axis and distance on the vertical axis reveals the **distance–risk trade-off**, not a smooth gradient.

A pre-rendered version of the chart is also available at `figures/risk_tradeoff.png`.


In [ ]:
weights = [0, .1, .2, .3, .32, .35, .5, 1, 2]
rows = []
for weight in weights:
    sol = solve_route(nodes, streets, closures, risk_weight=weight, backend=BACKEND)
    rows.append({'Risk weight':weight, 'Distance (m)':sol.distance_m,
                 'Exposure (risk × m)':round(sol.risk_exposure_m,2),
                 'Objective':round(sol.objective,2)})
sensitivity = pd.DataFrame(rows)
display(sensitivity)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sensitivity['Exposure (risk × m)'], sensitivity['Distance (m)'],
        '-o', lw=2)
for _, r in sensitivity.iterrows():
    if r['Risk weight'] in {0, .32}:
        ax.annotate(f"λ={r['Risk weight']:g}",
                    xy=(r['Exposure (risk × m)'],r['Distance (m)']),
                    xytext=(8, 8), textcoords='offset points')
ax.set_xlabel('Risk exposure (risk × m)')
ax.set_ylabel('Total route distance (m)')
ax.set_title('Sensitivity to risk aversion')
ax.grid(alpha=.2)
plt.show()

## 6. Model validation and takeaways

Verification beyond solver status:
- Route starts and ends at the operations hub.
- All five essential service stops are included.
- The route has no repeated non-depot junctions.
- Each **undirected street** is used at most once.

**Main takeaway:** A risk-aware route can incur a modest distance premium while reducing a modeled exposure index substantially. Storm closures can change the feasible network and demand a new solution. But the risk data are fabricated: any deployment requires validated hazard layers, vehicle capacity, service times, travel-time forecasts, and operational constraints.

**Portfolio integrity:** The fictional dataset, routes, and analysis here were independently created as a reusable example with AI assistance. No instructor source files are redistributed.

In [ ]:
for sol in [fast, safer, flash]:
    assert sol.route[0] == sol.route[-1] == 'DEPOT'
    assert len(set(sol.route[:-1])) == len(sol.route) - 1
    assert set(nodes.loc[nodes.is_required == 1, 'node_id']) <= set(sol.route)
print('Validation passed for all three selected routes.')